In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("uciml/glass")
file_path = os.path.join(path, "glass.csv")

df = pd.read_csv(file_path)
df.head()

c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


# 1. Pré-processamento e Limpeza dos Dados

Nesta etapa, serão realizadas verificações e tratamentos nos dados, incluindo:
- Remoção de duplicatas
- Análise de valores ausentes
- Tratamento de outliers
- Padronização dos dados

In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("uciml/glass")
file_path = os.path.join(path, "glass.csv")

df = pd.read_csv(file_path)

df.head()

c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB


## Verificação se há duplicadas
Como ocorre: A função duplicated verifica se há duplicação, caso seja True equivale a 1 e caso seja False equivale a 0. E o SUM() é uma função no python que soma tudo

In [3]:
duplicados = df.duplicated().sum()
print("Número de duplicatas:", duplicados)



Número de duplicatas: 1


O resultado mostra que tem duas linhas iguais

In [4]:
df = df.drop_duplicates() # remorção da duplicata



In [5]:
duplicados = df.duplicated().sum()
print(duplicados)



0


## Remorção de colunas com alta taxa de valores ausentes (threshold >
50%).


verifica se a coluna tá vazia, calcula proporção de True/(True+False) e multiplica vezes 100 para transformar em porcentagem

In [6]:
missing_percent = df.isnull().mean() * 100 
print(missing_percent)

RI      0.0
Na      0.0
Mg      0.0
Al      0.0
Si      0.0
K       0.0
Ca      0.0
Ba      0.0
Fe      0.0
Type    0.0
dtype: float64


ou seja: Em todas as colunas não há nenhum valor ausente

## Imputação de Valores Faltantes

A etapa de imputação de valores faltantes não foi necessária, uma vez que o dataset não apresenta valores ausentes em nenhuma de suas colunas.

Caso houvesse valores faltantes, poderiam ser utilizadas estratégias como média, mediana ou moda, dependendo da distribuição dos dados.

## Tratamento de Outliers

Foi utilizado o método Z-score para identificação de outliers, seguindo a abordagem apresentada no material da disciplina.

O Z-score mede quantos desvios padrão um valor está distante da média da variável. Neste trabalho, foram considerados possíveis outliers os valores com Z-score absoluto maior que 2.

A identificação de outliers é importante porque valores muito distantes da distribuição principal podem influenciar o treinamento da MLP, dificultando a convergência e afetando os ajustes dos pesos da rede neural.

In [7]:
# ============================
# 1. Separar features (sem o target)
# ============================

# Remove a coluna 'Type' (target), pois não analisamos outliers na variável alvo
features = df.drop("Type", axis=1)


# ============================
# 2. Calcular o Z-score
# ============================

# Aplica a fórmula do Z-score:
# (valor - média) / desvio padrão
z_scores = (features - features.mean()) / features.std()


# ============================
# 3. Identificar valores normais
# ============================

# Mantemos apenas valores cujo Z-score está entre -2 e 2
# abs() pega o valor absoluto (considera extremos positivos e negativos)
# <= 2 → valores normais
filtro = (z_scores.abs() <= 2)


# ============================
# 4. Remover linhas com outliers
# ============================

# .all(axis=1) → verifica linha por linha:
# mantém apenas linhas onde TODAS as colunas estão dentro do limite
df_clean = df[filtro.all(axis=1)]


# ============================
# 5. Verificar impacto
# ============================

print("Formato original:", df.shape)
print("Formato após remoção:", df_clean.shape)


# ============================
# 6. Atualizar dataset
# ============================

# Agora você passa a usar o dataset limpo
df = df_clean

Formato original: (213, 10)
Formato após remoção: (160, 10)


## Tratamento de Outliers

Foi utilizado o método Z-score para identificação de outliers, considerando valores com Z-score absoluto maior que 2.

As amostras que apresentaram valores discrepantes em qualquer uma das variáveis foram removidas do dataset. Essa decisão foi adotada para evitar a influência de valores extremos no treinamento da rede neural.

A remoção de outliers contribui para melhorar a estabilidade do modelo e favorecer a convergência da MLP, uma vez que reduz o impacto de valores anormais nos ajustes de peso.

## Normalização / Padronização dos Dados

Foi utilizada a técnica de padronização com StandardScaler, baseada no método Z-score, conforme apresentado no material da disciplina.

Essa abordagem transforma os dados para que apresentem média igual a 0 e desvio padrão igual a 1.

A padronização é essencial para redes neurais, como a MLP, pois essas são sensíveis à escala das variáveis de entrada. Quando as features possuem escalas diferentes, variáveis com valores maiores podem dominar o processo de aprendizado, prejudicando o desempenho do modelo.

Com a padronização, todas as variáveis passam a contribuir de forma equilibrada para o treinamento, favorecendo uma melhor convergência e estabilidade do modelo.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================
# 1. Separar variáveis
# ============================

# X = variáveis de entrada (features)
X = df.drop("Type", axis=1)

# y = variável alvo (target)
y = df["Type"]


# 2. Dividir treino e teste

# 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 3. Aplicar padronização


# Cria o scaler (baseado em Z-score)
scaler = StandardScaler()

# Aprende a escala com os dados de treino e transforma
X_train = scaler.fit_transform(X_train)

# Aplica a mesma transformação nos dados de teste
X_test = scaler.transform(X_test)

# ============================
# Fim da normalização
# ============================

In [9]:
# ============================================
# 2. Seleção de Variáveis - Método 1
# Correlação de Pearson com a variável target
# ============================================

# Separa as features e o target
X = df.drop("Type", axis=1)
y = df["Type"]

# Calcula a correlação de Pearson de cada feature com o target
correlacao_target = df.corr()["Type"].drop("Type")

# Usa o valor absoluto porque tanto correlação positiva quanto negativa podem ser importantes
correlacao_abs = correlacao_target.abs().sort_values(ascending=False)

print("Correlação de Pearson com o target:")
print(correlacao_abs)

Correlação de Pearson com o target:
Mg    0.715751
Ba    0.475293
Na    0.468343
Al    0.466210
K     0.395944
Ca    0.286263
Si    0.173040
Fe    0.159564
RI    0.069403
Name: Type, dtype: float64


### Correlação de Pearson

A análise de correlação de Pearson mostrou que a variável Mg apresentou a maior correlação com o target (0.71), indicando forte relação linear.

Outras variáveis como Ba, Na e Al também apresentaram correlações moderadas, contribuindo para a previsão do tipo de vidro.

Por outro lado, variáveis como RI, Fe e Si apresentaram baixa correlação, sugerindo menor relevância para o modelo.

In [10]:
# ============================================
# Método 2 - Mutual Information
# ============================================

from sklearn.feature_selection import mutual_info_classif

# calcula mutual information
mi = mutual_info_classif(X, y, random_state=42)

# transforma em série para visualizar melhor
mi_series = pd.Series(mi, index=X.columns)

# ordena
mi_sorted = mi_series.sort_values(ascending=False)

print("Mutual Information:")
print(mi_sorted)

Mutual Information:
Al    0.333675
RI    0.332833
K     0.318177
Mg    0.294219
Ca    0.232579
Na    0.190499
Ba    0.119837
Si    0.088290
Fe    0.029618
dtype: float64


### Mutual Information

A análise por Mutual Information revelou padrões diferentes em relação à correlação de Pearson, evidenciando relações não lineares entre as variáveis e o target.

Destaca-se a variável RI, que apresentou baixa correlação linear, mas alta informação mútua, indicando forte relação não linear com o tipo de vidro.

Variáveis como Al, K e Mg também apresentaram alta relevância, reforçando sua importância no modelo.

Por outro lado, a variável Fe apresentou baixa contribuição, sugerindo pouca relevância para a predição.

In [11]:
# ============================================
# Método 3 - Random Forest
# ============================================

from sklearn.ensemble import RandomForestClassifier

# cria o modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# treina
rf.fit(X, y)

# pega importância das variáveis
rf_importance = pd.Series(rf.feature_importances_, index=X.columns)

# ordena
rf_sorted = rf_importance.sort_values(ascending=False)

print("Importância Random Forest:")
print(rf_sorted)

Importância Random Forest:
RI    0.201995
Al    0.174756
Mg    0.151806
Ca    0.141165
K     0.095926
Si    0.094900
Na    0.086267
Fe    0.036452
Ba    0.016734
dtype: float64


## COMBINAR OS 3 MÉTODOS

In [12]:
# ============================================
# Normalizar os scores (0 a 1)
# ============================================

# correlação já estava em correlacao_target
corr = correlacao_target.abs()

corr_norm = (corr - corr.min()) / (corr.max() - corr.min())

mi_norm = (mi_series - mi_series.min()) / (mi_series.max() - mi_series.min())

rf_norm = (rf_importance - rf_importance.min()) / (rf_importance.max() - rf_importance.min())


# ============================================
# Criar tabela comparativa
# ============================================

df_features = pd.DataFrame({
    "correlacao": corr_norm,
    "mutual_info": mi_norm,
    "random_forest": rf_norm
})


# ============================================
# Score final (média)
# ============================================

df_features["score_final"] = df_features.mean(axis=1)


# ============================================
# Ranking final
# ============================================

ranking_final = df_features.sort_values("score_final", ascending=False)

print(ranking_final)

    correlacao  mutual_info  random_forest  score_final
Mg    1.000000     0.870238       0.729089     0.866442
Al    0.613921     1.000000       0.852967     0.822296
RI    0.000000     0.997232       1.000000     0.665744
K     0.505209     0.949032       0.427461     0.627234
Ca    0.335515     0.667511       0.671651     0.558226
Na    0.617222     0.529116       0.375328     0.507222
Ba    0.627975     0.296718       0.000000     0.308231
Si    0.160343     0.192966       0.421924     0.258411
Fe    0.139492     0.000000       0.106435     0.081976


### Ranking Final das Features

A combinação dos três métodos resultou em um ranking consolidado de importância das variáveis.

As variáveis Mg e Al apresentaram maior relevância, seguidas por RI e K. Por outro lado, a variável Fe apresentou o menor score, sendo considerada pouco relevante para o modelo.

A partir desse ranking, foi possível identificar quais variáveis contribuem mais significativamente para a predição do tipo de vidro.

In [13]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X.copy()

vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data.sort_values("VIF", ascending=False))

  feature            VIF
0      RI  816561.159037
4      Si  415727.958505
1      Na   15780.508653
6      Ca    8724.370298
2      Mg    1083.234817
3      Al     205.410774
5       K      41.369551
7      Ba       3.520423
8      Fe       1.468233


### Tratamento de Multicolinearidade (VIF)

A análise de VIF revelou valores extremamente elevados para diversas variáveis, indicando forte multicolinearidade no dataset.

Esse comportamento é esperado, pois os atributos representam componentes químicos cuja soma é aproximadamente constante, gerando dependência entre as variáveis.

Dessa forma, optou-se por remover variáveis com menor relevância no ranking final e alta redundância, como Fe, Si e Ba.

As variáveis restantes foram selecionadas com base na combinação entre importância e redução da multicolinearidade, preservando aquelas com maior contribuição para o modelo.

In [14]:
# ============================================
# Aplicar seleção de features
# ============================================

# lista final escolhida
features_selecionadas = ["Mg", "Al", "RI", "K", "Ca", "Na"]

# criar novo dataset só com essas colunas + target
df = df[features_selecionadas + ["Type"]]

# verificar resultado
print(df.head())
print("\nColunas finais:", df.columns)

     Mg    Al       RI     K    Ca     Na  Type
0  4.49  1.10  1.52101  0.06  8.75  13.64     1
1  3.60  1.36  1.51761  0.48  7.83  13.89     1
2  3.55  1.54  1.51618  0.39  7.78  13.53     1
3  3.69  1.29  1.51766  0.57  8.22  13.21     1
4  3.62  1.24  1.51742  0.55  8.07  13.27     1

Colunas finais: Index(['Mg', 'Al', 'RI', 'K', 'Ca', 'Na', 'Type'], dtype='str')


### Seleção Final de Features

Com base no ranking combinado e na análise de multicolinearidade (VIF), foram selecionadas as variáveis mais relevantes para o modelo: Mg, Al, RI, K, Ca e Na.

As variáveis Fe, Si e Ba foram removidas por apresentarem baixa importância e/ou alta redundância em relação às demais.

Essa seleção visa reduzir a complexidade do modelo, minimizar problemas de multicolinearidade e manter apenas as informações mais relevantes para a predição.

### PArte 2

## Preparar dados para a MLP

In [16]:
# ============================================
# Preparação dos dados para a MLP
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X = features (sem o target)
X = df.drop("Type", axis=1)

# y = target
y = df["Type"]

# divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# padronização (ESSENCIAL pra MLP)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Dados prontos!")
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Dados prontos!
Treino: (128, 6)
Teste: (32, 6)


In [17]:
# ============================================
# MLP - modelo base
# ============================================

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

# modelo inicial simples
mlp = MLPClassifier(
    hidden_layer_sizes=(16,),   # 1 camada com 16 neurônios
    activation='relu',
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42
)

# treinar
mlp.fit(X_train, y_train)

# prever
y_pred = mlp.predict(X_test)

# métricas
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("Acurácia:", acc)
print("F1-score:", f1)

Acurácia: 0.59375
F1-score: 0.5339586255259468


c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [18]:
# ============================================
# Comparação empírica de hiperparâmetros da MLP
# ============================================

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

configuracoes = [
    # 1 camada oculta
    {"hidden_layer_sizes": (8,), "activation": "relu", "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (16,), "activation": "relu", "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32,), "activation": "relu", "learning_rate_init": 0.001},

    # 2 camadas ocultas
    {"hidden_layer_sizes": (16, 8), "activation": "relu", "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32, 16), "activation": "relu", "learning_rate_init": 0.001},

    # comparar funções de ativação
    {"hidden_layer_sizes": (16,), "activation": "tanh", "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (16,), "activation": "logistic", "learning_rate_init": 0.001},

    # comparar learning rate
    {"hidden_layer_sizes": (16,), "activation": "relu", "learning_rate_init": 0.01},
    {"hidden_layer_sizes": (16,), "activation": "relu", "learning_rate_init": 0.0001},
]

resultados = []

for config in configuracoes:
    mlp = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        activation=config["activation"],
        learning_rate_init=config["learning_rate_init"],
        max_iter=2000,          # aumentamos para dar mais chance de convergir
        random_state=42
    )

    mlp.fit(X_train, y_train)

    y_pred = mlp.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    resultados.append({
        "camadas_ocultas": config["hidden_layer_sizes"],
        "ativacao": config["activation"],
        "learning_rate": config["learning_rate_init"],
        "accuracy": acc,
        "f1_score": f1,
        "iteracoes_usadas": mlp.n_iter_
    })

resultados_df = pd.DataFrame(resultados)
resultados_df.sort_values("f1_score", ascending=False)

c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\SOL

,camadas_ocultas,ativacao,learning_rate,accuracy,f1_score,iteracoes_usadas
3,"(16, 8)",relu,0.0010,0.78125,0.779018,1845
4,"(32, 16)",relu,0.0010,0.75000,0.752459,1146
7,"(16,)",relu,0.0100,0.71875,0.728854,864
1,"(16,)",relu,0.0010,0.71875,0.721786,2000
2,"(32,)",relu,0.0010,0.71875,0.692361,2000
5,"(16,)",tanh,0.0010,0.68750,0.680151,2000
0,"(8,)",relu,0.0010,0.59375,0.533959,2000
8,"(16,)",relu,0.0001,0.59375,0.533959,2000
6,"(16,)",logistic,0.0010,0.56250,0.513084,2000


In [19]:
# ============================================
# Teste de regularização (simulando dropout)
# ============================================

alphas = [0.0001, 0.001, 0.01]

resultados_alpha = []

for alpha in alphas:
    mlp = MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation='relu',
        learning_rate_init=0.001,
        alpha=alpha,   # regularização
        max_iter=2000,
        random_state=42
    )

    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    resultados_alpha.append({
        "alpha": alpha,
        "accuracy": acc,
        "f1_score": f1
    })

pd.DataFrame(resultados_alpha)

c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\SOLTEC\glass-ml-project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


,alpha,accuracy,f1_score
0,0.0001,0.78125,0.779018
1,0.0010,0.78125,0.779018
2,0.0100,0.78125,0.779018


## 3. Definição e Justificativa dos Hiperparâmetros da MLP

Para esta etapa, foi utilizada uma Rede Neural do tipo MLP (Multilayer Perceptron), com o objetivo de avaliar o impacto de diferentes hiperparâmetros no desempenho do modelo.

### Número de camadas ocultas (profundidade)

Foram testadas arquiteturas com 1 e 2 camadas ocultas. Observou-se que modelos com duas camadas ocultas apresentaram melhor desempenho em relação aos modelos com apenas uma camada.

Isso ocorre porque o aumento da profundidade permite maior capacidade de representação de padrões complexos. No entanto, modelos muito profundos podem levar ao overfitting. Para dados tabulares, como neste caso, a literatura sugere que 1 a 3 camadas são suficientes.

### Número de neurônios por camada (largura)

Foram testadas diferentes quantidades de neurônios (8, 16 e 32). Observou-se que:

- Poucos neurônios (8) resultaram em baixo desempenho (underfitting);
- Quantidades intermediárias (16) apresentaram melhor equilíbrio;
- Aumentos adicionais (32) não trouxeram ganhos significativos.

Dessa forma, foi escolhido um número intermediário de neurônios para evitar complexidade desnecessária.

### Funções de ativação

Foram testadas diferentes funções de ativação:

- ReLU (Rectified Linear Unit)
- Tanh
- Sigmoid (logistic)

A função ReLU apresentou melhor desempenho. Isso se deve ao fato de que a ReLU evita o problema do gradiente desaparecendo e possui menor custo computacional, sendo a escolha padrão para camadas ocultas em redes neurais.

Já as funções Tanh e Sigmoid apresentaram desempenho inferior, possivelmente devido à saturação dos gradientes.

Para a camada de saída, foi utilizada implicitamente a função adequada para classificação multiclasse, com base na função de custo utilizada (Cross-Entropy).

### Taxa de aprendizado (learning rate)

Foram testados diferentes valores:

- 0.0001 → aprendizado muito lento
- 0.001 → melhor desempenho
- 0.01 → instabilidade no treinamento

Observou-se que valores muito baixos dificultam a convergência, enquanto valores muito altos podem causar oscilações. Assim, foi escolhido o valor intermediário (0.001).

### Regularização (Dropout / Alpha)

Como a implementação utilizada (scikit-learn) não possui dropout direto, foi utilizada a regularização L2 (parâmetro alpha) como alternativa.

Foram testados diferentes valores de alpha, porém não houve variação significativa no desempenho do modelo. Isso indica que o modelo não estava sofrendo overfitting relevante.

### Função de custo (Loss)

A função de custo utilizada foi a Cross-Entropy (implementada internamente no MLPClassifier). Essa função é adequada para problemas de classificação multiclasse, pois mede a diferença entre as probabilidades previstas e as classes reais.

### Modelo final selecionado

Com base nos experimentos realizados, o modelo final escolhido foi:

- Camadas ocultas: (16, 8)
- Função de ativação: ReLU
- Taxa de aprendizado: 0.001
- Regularização (alpha): 0.0001

Esse modelo apresentou o melhor desempenho geral, equilibrando capacidade de aprendizado e estabilidade.